In [9]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


cwd: c:\Users\okofoworola\Sutter Health Demo\fine-tuning


# Lab 18b · Multimodal Imaging Flow (virtual tumor board)

Lab 18 wired a *triage → specialists* **team** for a pharmacy scenario. Oncology research is imaging-heavy, so this variant retargets the **same pattern** at a lightweight **virtual tumor board**:

- **Perception (multimodal).** A vision model (`gpt-4o`) reads an actual scan image and writes a radiology-style impression.
- **Reasoning team (flow).** A **case-coordinator** agent fans the case out to two server-side specialists via `ConnectedAgentTool` — an **imaging-findings** specialist and a **prior-comparison** specialist (RECIST 1.1 response vs a baseline) — then merges a tumor-board draft.

**Why split perception from the team?** Connected sub-agents run **server-side**; feeding raw image bytes into them is fragile, and client-side function tools make the run *hang* (Lab 18's hard-won lesson). So we run the multimodal read **once, up front**, and feed its **text** into an instruction-driven, server-side flow — exactly how real systems work: a segmentation / PACS tool supplies the measurement; the LLM team reasons about it.

> ⚠️ **Safety & scope.** The scan below is **synthetic** (numpy/PIL — no PHI). Vision LLMs here are **research / decision-support, not a diagnostic device**. Keep a clinician in the loop for anything patient-facing.


---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every agent call below shows up live in the Microsoft Foundry portal under **your project → Tracing**. For a tumor-board flow the trace is your audit trail — *which* specialist saw *what*, in what order — exactly the record clinical / IRB reviewers ask for.*


In [10]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='sutter-imaging-flow-lab')


[tracing] already enabled (service=sutter-imaging-flow-lab) - skipping.


True

---
## Step 1 — Connect: Agent Service + a vision model

We need two clients: the **Foundry Agent Service** (to build the specialist team) and an **Azure OpenAI** client pointed at a vision-capable deployment (`gpt-4o`) for the multimodal read. Both use the same `az login` / managed-identity credential and degrade gracefully if a piece is missing.


In [11]:
import os
from _advisor import get_credential, try_build_client

ACCT    = os.environ.get('AZURE_RESOURCE_NAME', 'aif-shuttervoice-dev')
PROJECT = os.environ.get('AZURE_FOUNDRY_PROJECT', 'agents')
PROJECT_ENDPOINT = f'https://{ACCT}.services.ai.azure.com/api/projects/{PROJECT}'
MODEL = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')        # text agents
VISION_MODEL = os.environ.get('VISION_DEPLOYMENT', 'gpt-4o')    # multimodal read

# Track every agent we create so cleanup (Step 6) never leaks resources.
created_agent_ids: list[str] = []

agents = None
try:
    from azure.ai.agents import AgentsClient
    agents = AgentsClient(endpoint=PROJECT_ENDPOINT, credential=get_credential())
    print('Agents client ready ->', PROJECT_ENDPOINT)
except Exception as e:
    print('[skip] Foundry Agent Service unavailable:', type(e).__name__, e)

# Vision-capable Azure OpenAI client (reused from the advisor helper).
vision = try_build_client()
print('Vision client:', 'ready ->' if vision is not None else 'unavailable (mock)',
      VISION_MODEL if vision is not None else '')


Agents client ready -> https://aif-shuttervoice-dev.services.ai.azure.com/api/projects/agents
Vision client: ready -> gpt-4o


---
## Step 2 — Synthesize a PHI-free sample scan

Real oncology images are PHI and can't ship in a demo repo. So we **generate** a CT-like slice: soft-tissue background, a body ellipse, and one bright **simulated lesion**. No patient data, fully reproducible. (In production this is a real DICOM slice pulled from PACS / a segmentation pipeline.)


In [13]:
import base64
SAMPLE_PNG = Path('data/samples/sample_ct_slice.png')
SAMPLE_PNG.parent.mkdir(parents=True, exist_ok=True)

img_b64 = None
try:
    import numpy as np
    from PIL import Image, ImageFilter

    rng = np.random.default_rng(42)
    H = W = 256
    yy, xx = np.mgrid[0:H, 0:W]
    img = rng.normal(38, 7, (H, W))                      # background noise
    body = (((xx - 128) / 110) ** 2 + ((yy - 128) / 95) ** 2) <= 1
    img[body] = rng.normal(95, 9, int(body.sum()))       # soft tissue
    # simulated target lesion (bright, well-circumscribed)
    cx, cy, r = 158, 104, 12
    lesion = np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * r ** 2)))
    img = np.clip(img + lesion * 125, 0, 255).astype('uint8')
    im = Image.fromarray(img, mode='L').filter(ImageFilter.GaussianBlur(0.6))
    im.save(SAMPLE_PNG)
    img_b64 = base64.b64encode(SAMPLE_PNG.read_bytes()).decode()
    print('Wrote synthetic scan ->', SAMPLE_PNG, f'({len(img_b64)} b64 chars)')
except Exception as e:
    print('[skip] could not synthesize image:', type(e).__name__, e)

# Structured measurement a segmentation/PACS tool would supply alongside the pixels.
CURRENT_LONGEST_DIAMETER_MM = 20   # this scan
BASELINE_LONGEST_DIAMETER_MM = 30  # prior study (for RECIST comparison)


Wrote synthetic scan -> data\samples\sample_ct_slice.png (51264 b64 chars)


---
## Step 3 — Multimodal perception: read the scan (LIVE)

`gpt-4o` looks at the **actual pixels** and writes a short, radiology-style impression (location, margins, enhancement). This is the one genuinely *multimodal* step — its **text** output becomes the input to the agent team. If the vision model or content filter blocks, we fall back to a canned impression so the flow still runs.


In [14]:
FALLBACK_IMPRESSION = (
    'Single well-circumscribed hyperdense focus in the right upper quadrant '
    'with smooth margins and mild homogeneous enhancement; no satellite '
    'lesions on this slice. Appearances are of a discrete measurable target lesion.'
)

impression = None
if vision is not None and img_b64:
    try:
        data_uri = f'data:image/png;base64,{img_b64}'
        resp = vision.chat.completions.create(
            model=VISION_MODEL,
            max_tokens=300,
            messages=[{
                'role': 'user',
                'content': [
                    {'type': 'text', 'text': (
                        'You are a radiology research assistant. This is a SYNTHETIC '
                        'CT-like slice (no patient data). Describe the single bright '
                        'focus: location, margins, enhancement, and whether it reads '
                        'as a discrete measurable target lesion. 3-4 sentences. '
                        'Do NOT give a diagnosis or clinical advice.')},
                    {'type': 'image_url', 'image_url': {'url': data_uri}},
                ],
            }],
        )
        impression = resp.choices[0].message.content.strip()
        print('--- vision impression (gpt-4o) ---')
    except Exception as e:
        print('[fallback] vision read unavailable:', type(e).__name__, e)

if not impression:
    impression = FALLBACK_IMPRESSION
    print('--- vision impression (fallback) ---')
print(impression)


--- vision impression (gpt-4o) ---
The bright focus is centrally located within the synthetic CT-like slice, slightly towards the upper right quadrant of the oval structure. It displays well-defined margins, appearing distinct from the surrounding tissue. The enhancement is significant, making it stand out prominently against the background. It can be considered a discrete measurable target lesion based on its clear delineation and contrast relative to adjacent areas.


---
## Step 4 — Create the specialist agents (server-side)

Two domain experts, each a plain Foundry agent with a tight scope:

- **Imaging-findings specialist** — turns the impression + measurement into a structured target-lesion summary.
- **Prior-comparison specialist** — computes the change vs the baseline and assigns a **RECIST 1.1** response category.

> **Why no client-side `FunctionTool`s here?** A connected sub-agent runs **server-side** inside the coordinator's run. Client-side function tools need *your process* to submit their outputs — a signal the coordinator run never surfaces, so the flow would hang. Inside a flow, specialists use **server-side capabilities** (instructions, `file_search`, Azure AI Search, code interpreter). The RECIST arithmetic is simple enough to bake into instructions; in production you'd hand it to the **code interpreter** tool for audited math.


In [16]:
findings_agent = compare_agent = None
if agents is not None:
    try:
        findings_agent = agents.create_agent(
            model=MODEL, name='sutter-imaging-findings-specialist',
            instructions=(
                'You are an oncology imaging-findings specialist. You receive a '
                'radiology impression and a measured longest-diameter (mm) for a '
                'target lesion. Produce a tight structured summary: lesion location, '
                'margins/enhancement (from the impression), and the current longest '
                'diameter in mm. Confirm whether it qualifies as a measurable RECIST '
                'target lesion (>= 10 mm). Be concise. Do NOT give a diagnosis or '
                'treatment advice; this is research decision-support.'))
        created_agent_ids.append(findings_agent.id)
        print('Imaging-findings specialist:', findings_agent.id)

        compare_agent = agents.create_agent(
            model=MODEL, name='sutter-prior-comparison-specialist',
            instructions=(
                'You are a RECIST 1.1 response specialist. You receive a current '
                'longest-diameter (mm) and a baseline longest-diameter (mm) for a '
                'single target lesion. Compute the percent change = '
                '(current - baseline) / baseline * 100, rounded to one decimal. '
                'Classify per RECIST 1.1 (single target-lesion approximation): '
                'Complete Response (CR) if current is 0; Partial Response (PR) if '
                'decrease >= 30%; Progressive Disease (PD) if increase >= 20%; '
                'otherwise Stable Disease (SD). State the percent change and the '
                'category with a one-line rationale. Research decision-support only.'))
        created_agent_ids.append(compare_agent.id)
        print('Prior-comparison specialist:', compare_agent.id)
    except Exception as e:
        print('[skip] could not create specialists:', type(e).__name__, e)
else:
    print('[skip] no agents client - see Step 1.')


Imaging-findings specialist: asst_UAx2obhnKxp5R3kuF71AaI3O
Prior-comparison specialist: asst_CCxj8baKonP9WvCMalzaBCJV


---
## Step 5 — Create the case-coordinator that connects the team

`ConnectedAgentTool(id, name, description)` turns each specialist into a **tool** the coordinator can call. The `description` is the routing signal — Foundry reads it to decide which specialist a case needs. No glue code: the platform owns the fan-out and merge.


In [17]:
coordinator = None
if findings_agent is not None and compare_agent is not None:
    try:
        from azure.ai.agents.models import ConnectedAgentTool

        find_conn = ConnectedAgentTool(
            id=findings_agent.id, name='imaging_findings_specialist',
            description='Summarizes the radiology impression and confirms the measurable target lesion.')
        comp_conn = ConnectedAgentTool(
            id=compare_agent.id, name='prior_comparison_specialist',
            description='Computes RECIST 1.1 response from current vs baseline lesion diameter.')

        connected_defs = list(find_conn.definitions) + list(comp_conn.definitions)
        coordinator = agents.create_agent(
            model=MODEL, name='sutter-tumorboard-coordinator',
            instructions=(
                'You are the case coordinator for a research tumor board. For each '
                'case, send the impression + current measurement to the imaging-findings '
                'specialist, and the current + baseline measurements to the '
                'prior-comparison specialist. Then merge both into ONE concise '
                'tumor-board draft: Findings, RECIST response, and a Suggested next '
                'step (e.g. continue surveillance / discuss at board). End with: '
                '"Research decision-support only - not a diagnosis."'),
            tools=connected_defs)
        created_agent_ids.append(coordinator.id)
        print('Tumor-board coordinator   :', coordinator.id)
        print('Connected specialists     :', 'imaging_findings_specialist, prior_comparison_specialist')
    except Exception as e:
        print('[skip] could not create coordinator:', type(e).__name__, e)
else:
    print('[skip] specialists missing - see Step 4.')


Tumor-board coordinator   : asst_eORvnoFT0kKLH5plyEabp4SN
Connected specialists     : imaging_findings_specialist, prior_comparison_specialist


---
## Step 6 — Run the case through the whole board (LIVE)

One case, two specialties: *describe the target lesion* (findings) **and** *is it responding vs baseline* (RECIST). The coordinator fans out to both connected agents, then merges the tumor-board draft. We read the **run steps** to visualise the flow — each connected-agent call is a node in the trace. The run is polled with a **hard 120 s timeout + cancel** so a flow can never hang the notebook.


In [18]:
import time
if coordinator is not None:
    try:
        case_msg = (
            'New surveillance scan for research case RC-4471.\n'
            f'Radiology impression: {impression}\n'
            f'Current target-lesion longest diameter: {CURRENT_LONGEST_DIAMETER_MM} mm.\n'
            f'Baseline (prior study) longest diameter: {BASELINE_LONGEST_DIAMETER_MM} mm.\n'
            'Please assess findings and RECIST response, then draft the board note.')
        thread = agents.threads.create()
        agents.messages.create(thread_id=thread.id, role='user', content=case_msg)
        # Poll explicitly with a hard timeout so the flow can never hang the notebook.
        run = agents.runs.create(thread_id=thread.id, agent_id=coordinator.id)
        t0 = time.time()
        while run.status in ('queued', 'in_progress') and time.time() - t0 < 120:
            time.sleep(2)
            run = agents.runs.get(thread_id=thread.id, run_id=run.id)
        if run.status in ('queued', 'in_progress'):
            agents.runs.cancel(thread_id=thread.id, run_id=run.id)
        print('Run status:', run.status, f'({int(time.time() - t0)}s)')
        if getattr(run, 'last_error', None):
            print('last_error:', run.last_error)

        print('\n--- tumor-board flow trace (run steps) ---')
        for s in agents.run_steps.list(thread_id=thread.id, run_id=run.id):
            det = getattr(s, 'step_details', None)
            calls = getattr(det, 'tool_calls', None) or []
            if calls:
                for tc in calls:
                    name = getattr(getattr(tc, 'connected_agent', None), 'name', None) \
                           or getattr(getattr(tc, 'function', None), 'name', None) \
                           or getattr(tc, 'type', '?')
                    print(f'  step {getattr(s, "type", "?"):<16} -> routed to: {name}')
            else:
                print(f'  step {getattr(s, "type", "?"):<16} -> {getattr(det, "type", "?")}')

        print('\n--- tumor-board draft ---')
        for m in agents.messages.list(thread_id=thread.id):
            if m.role == 'assistant':
                for c in m.content:
                    if getattr(c, 'text', None):
                        print('BOARD:', c.text.value)
                break
    except Exception as e:
        print('[skip] flow run failed:', type(e).__name__, e)
else:
    print('[skip] no coordinator - see Step 5.')


Run status: RunStatus.COMPLETED (19s)

--- tumor-board flow trace (run steps) ---
  step RunStepType.MESSAGE_CREATION -> message_creation
  step RunStepType.TOOL_CALLS -> routed to: connected_agent
  step RunStepType.TOOL_CALLS -> routed to: connected_agent

--- tumor-board draft ---
BOARD: ### Tumor Board Draft for Case RC-4471

**Findings:**  
The imaging reveals a bright focus in the upper right quadrant of the oval structure, characterized by well-defined margins and significant enhancement, distinguishing it from surrounding tissue. This lesion measures 20 mm and is considered a discrete measurable target lesion.

**RECIST Response:**  
The current longest diameter of the target lesion is 20 mm, down from a baseline measurement of 30 mm. This represents a 33.3% decrease, qualifying as a Partial Response (PR) as per RECIST 1.1 criteria.

**Suggested Next Step:**  
Continue surveillance and reassess in subsequent imaging.

*Research decision-support only - not a diagnosis.*


---
## Step 7 — Clean up the whole board

Every agent we created — coordinator **and** both specialists — is a persistent resource. Delete them all so only the seeded production agents remain. (We tracked each id in `created_agent_ids` precisely for this.)


In [9]:
if agents is not None and created_agent_ids:
    for aid in created_agent_ids:
        try:
            agents.delete_agent(aid)
            print('Deleted agent', aid)
        except Exception as e:
            print('[warn] cleanup', aid, ':', e)
else:
    print('Nothing to clean up.')


Deleted agent asst_528UzCF4wGlUpdEICzjTJiAw


Deleted agent asst_Wfq2wdPFtwLK62toawhvx54Z


Deleted agent asst_tWMCUvt6KVrfe4vXIlWcpghB


---
## Step 8 — Production path for imaging (reference)

To take this from demo to research-grade, swap the stubs for server-side tools the connected specialists can use **without hanging**:

- **Segmentation / measurement** → a dedicated imaging endpoint (e.g. a **MONAI** model on Azure ML or Container Apps) the coordinator calls as a server-side tool; it returns lesion masks + RECIST diameters instead of our hard-coded mm.
- **Prior studies & literature** → **Azure AI Search** over an indexed corpus of de-identified prior reports / trial criteria, attached to a specialist via `AzureAISearchTool` (server-side `file_search` works the same way for uploaded PDFs).
- **Audited arithmetic** → the **code interpreter** tool for the RECIST math, so every number has a reproducible cell behind it.
- **Visual design & versioning** → build the board as a **Foundry Workflow** in the portal canvas and invoke it by name (`azure-ai-projects>=2.1.0`), exactly as Lab 18 Step 6 shows.

**Governance.** Synthetic data here → real workflows need PHI handling (Lab 10), groundedness / RAI checks (Lab 17), and a human-in-the-loop sign-off. Treat every agent output as **decision-support**, never an autonomous diagnosis.


---
## Takeaways

- The Lab 18 **flow pattern transfers wholesale** to oncology imaging — just rename the specialists and feed them imaging context.
- **Perception and reasoning are separate tiers:** do the multimodal read once (`gpt-4o` on the pixels), then route its **text** through a server-side specialist team. That keeps the flow fast and hang-proof.
- `ConnectedAgentTool` makes the board **declarative** — add a radiogenomics or trial-matching specialist later without touching the coordinator.
- The **run steps** are your tumor-board audit trail (Step 0 tracing) — the *who-saw-what* record reviewers expect.
- Production = **server-side tools** (segmentation endpoint, Azure AI Search, code interpreter) + a portal **Workflow**, with PHI / RAI / human-in-the-loop governance.

*Related: Lab 11 (single agent + tools) → Lab 18 (multi-agent flow) → Lab 18b (multimodal imaging flow).*
